# Explicar código — 2SLS matricial

**Unidad 4.c** · Acompaña a `Clase_02_VariablesInstrumentales` · Notas: cap. 3

A diferencia de las actividades de `02_Depurar`, aquí **no hay ningún error**. El código
es correcto y replica a Acemoglu, Johnson y Robinson (2001), cuadro 4. Lo que tiene de
malo es que no se entiende.

> El riesgo específico de esta actividad: **un asistente de IA explica muy bien el código
> correcto**, y eso produce una sensación de comprensión que no siempre corresponde a
> comprensión. La explicación se valida contra el álgebra, que es lo único que decide si
> el código implementa el estimador que se quería.

In [1]:
import numpy as np
import pandas as pd

# Acemoglu, Johnson y Robinson (2001), cuadro 4, panel A, columna 1.
PUBLICADO_2SLS = 0.94
PUBLICADO_N = 64

datos = pd.read_stata("../../Clase_02_VariablesInstrumentales/maketable4.dta")
datos = datos[datos["baseco"] == 1].dropna(subset=["logpgp95", "avexpr", "logem4"])

print(f"Muestra base: {len(datos)} países")
datos[["shortnam", "logpgp95", "avexpr", "logem4"]].head()

Muestra base: 64 países


,shortnam,logpgp95,avexpr,logem4
1,AGO,7.770645,5.363636,5.634789
3,ARG,9.133459,6.386364,4.232656
5,AUS,9.897972,9.318182,2.145931
11,BFA,6.845880,4.454545,5.634789
12,BGD,6.877296,5.136364,4.268438


## El fragmento que hay que explicar

Seis líneas, sin un solo comentario. **No lo modifiques hasta el final de la actividad.**

In [2]:
def estimar(d):
    y = d["logpgp95"].values
    n = len(d)
    X = np.column_stack([np.ones(n), d["avexpr"].values])
    Z = np.column_stack([np.ones(n), d["logem4"].values])
    P = Z @ np.linalg.solve(Z.T @ Z, Z.T)
    b = np.linalg.solve(X.T @ P @ X, X.T @ P @ y)
    e = y - X @ b
    s2 = (e @ e) / (n - X.shape[1])
    V = s2 * np.linalg.inv(X.T @ P @ X)
    return b, np.sqrt(np.diag(V)), P, X, Z, n


b, ee, P, X, Z, n = estimar(datos)

print(f"N = {n}\n")
print(f"  constante : {b[0]:8.4f}   (ee {ee[0]:.4f})")
print(f"  avexpr    : {b[1]:8.4f}   (ee {ee[1]:.4f})")

N = 64

  constante :   1.9097   (ee 1.0267)
  avexpr    :   0.9443   (ee 0.1565)


## Propiedades que cumple la matriz `P`

El código construye un objeto `P` cuyo papel no es obvio a primera vista. Estas tres
propiedades lo identifican; hay que poder decir de dónde salen y por qué son ésas.

In [3]:
print(f"  simétrica  (P = P')  : {np.allclose(P, P.T)}")
print(f"  idempotente (PP = P) : {np.allclose(P @ P, P)}")
print(f"  rango                : {np.linalg.matrix_rank(P)}  (columnas de Z: {Z.shape[1]})")
print(f"  PZ = Z               : {np.allclose(P @ Z, Z)}")

  simétrica  (P = P')  : True
  idempotente (PP = P) : True
  rango                : 2  (columnas de Z: 2)
  PZ = Z               : True


## Verificación contra el artículo publicado

In [4]:
print(f"  2SLS estimado  : {b[1]:.4f}")
print(f"  2SLS publicado : {PUBLICADO_2SLS:.4f}")
print(f"  N estimado {n}, publicado {PUBLICADO_N}")
print()
print("  REPLICA." if abs(b[1] - PUBLICADO_2SLS) < 0.01 else "  NO REPLICA — revisar.")

  2SLS estimado  : 0.9443
  2SLS publicado : 0.9400
  N estimado 64, publicado 64

  REPLICA.


## Por qué importa el instrumento

El contraste con MCO es el punto sustantivo del capítulo 3.

In [5]:
b_mco = np.linalg.solve(X.T @ X, X.T @ datos["logpgp95"].values)

print(f"  MCO  : {b_mco[1]:.4f}")
print(f"  2SLS : {b[1]:.4f}")
print(f"  2SLS es {b[1] / b_mco[1]:.2f} veces mayor que MCO.")

  MCO  : 0.5221
  2SLS : 0.9443
  2SLS es 1.81 veces mayor que MCO.


Además conviene medir la fuerza del instrumento. La regla práctica de Staiger y Stock
(1997) pide un estadístico $F$ de primera etapa por encima de 10.

In [6]:
import statsmodels.api as sm

primera = sm.OLS(datos["avexpr"], Z).fit()
print(f"Primera etapa: coef de logem4 = {primera.params.iloc[1]:.4f}")
print(f"               t = {primera.tvalues.iloc[1]:.2f}")
print(f"               F = {primera.fvalue:.2f}", end="")
print("   (por encima de 10: el instrumento no es débil)")

Primera etapa: coef de logem4 = -0.6068
               t = -4.79
               F = 22.95   (por encima de 10: el instrumento no es débil)


## Tareas

1. Lee `estimar()` e intenta decir en una frase qué hace **cada línea**. Anota en cuáles
   te quedas sin poder explicarlo.
2. Pídele a un asistente de IA que lo explique línea por línea.
3. **Verifica la explicación contra el álgebra del capítulo 3**, no contra tu intuición
   ni contra lo convincente que suene. En particular:
   - ¿Qué objeto es `P`? ¿Por qué las tres propiedades de arriba y no otras?
   - ¿Por qué `np.linalg.solve(A, b)` y no `np.linalg.inv(A) @ b`?
   - La fórmula que implementa, ¿es la ecuación de 2SLS de las notas o una equivalente?
     Demuéstralo.
   - ¿Por qué `s2` divide entre `n - 2` y no entre `n`?
4. Reescribe la función con nombres y comentarios que la vuelvan legible, **sin cambiar
   lo que calcula**. Las comprobaciones de arriba deben seguir pasando.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las demás actividades.

In [7]:
# Tu versión legible aquí.